# Lección 11.2 — Representar palabras: tokens y *embeddings* (el mapa de significados)

> **Módulo 11 · Secuencias y lenguaje** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.3 (one-hot), Lección 5.3 (KNN y distancias), Lección 9.4 (retropropagación), Lección 11.1

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es un **token** y por qué los modelos modernos cortan las palabras en pedazos.
- Explicar por qué el **one-hot** es una mala forma de representar palabras.
- Describir un ***embedding*** como un **mapa de significados** y cómo se aprende ("dime con quién andas…").
- Entrenar unos *embeddings* pequeños y explorar los de un modelo real (**GPT-2**).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez. La primera vez carga GPT-2 (≈500 MB); si no lo tienes en caché, se descargará (en Colab tarda ~1 min).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModelForCausalLM

plt.rcParams["figure.figsize"] = (7, 5)
torch.manual_seed(0)

tokenizador = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
print("✅ Todo listo")

---
## 1. 📖 La intuición: un mapa de significados

Imagina un mapa de una ciudad donde cada **palabra** vive en una dirección:

```
            🍎 manzana  🍐 pera
                  🍇 uva                       🚗 coche   🚌 autobús
                                                   🚚 camión
      🐶 perro  🐱 gato
         🐴 caballo                ☀️ lunes  martes  miércoles
```

Las palabras **parecidas viven cerca** (las frutas en un barrio, los días en otro) y las muy distintas viven lejos. Con un mapa así, una computadora puede "saber" que *perro* se parece más a *gato* que a *camión*, sin que nadie se lo diga.

Un ***embedding*** (en español a veces "incrustación" o "vector de palabra") es exactamente eso: **la dirección de una palabra en un mapa de significados**. La diferencia es que el mapa no tiene 2 dimensiones, sino decenas o cientos (GPT-2 usa 768).

### ✋ Ejercicio 1 — Dibuja tu propio mapa
En un mapa 2D como el de arriba, ¿dónde pondrías **"naranja"**? ¿Y **"tren"**? ¿Y **"jaguar"** (que puede ser un animal o una marca de coches)?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Naranja** → en el barrio de las frutas (aunque también es un color: ¡ojo con eso!).
- **Tren** → cerca de los vehículos.
- **Jaguar** → ¡problema! Debería estar cerca de los animales **y** de los coches a la vez. En un mapa donde cada palabra tiene **una sola** dirección, una palabra con dos significados queda "a medio camino". Volveremos a este problema en la Sección 6.
</details>

---
## 2. 📖 Paso 1: cortar el texto en *tokens*

Antes de darle direcciones a las palabras, hay que decidir **qué es una "palabra"**. A cada pedazo de texto que el modelo trata como una unidad se le llama **token**, y al proceso de cortar, **tokenización**.

| Estrategia | "los murciélagos" se corta en… | ✅ Ventaja | ❌ Problema |
|---|---|---|---|
| Por **caracteres** | `l o s ␣ m u r c i é l a g o s` | Vocabulario minúsculo (~100) | Secuencias larguísimas |
| Por **palabras** | `los · murciélagos` | Secuencias cortas | Vocabulario enorme; palabras nuevas = desconocidas |
| Por **subpalabras** | `los · murci · élag · os` (aprox.) | Lo mejor de ambos | Cortes raros, distintos por idioma |

Los modelos modernos (GPT, BERT, Claude…) usan **subpalabras**: las palabras frecuentes son un token entero y las raras se arman con pedazos. Así **nunca** hay una palabra "desconocida": en el peor caso se arma letra por letra.

Veamos cómo corta el tokenizador de **GPT-2** (entrenado sobre todo con texto en inglés):

In [ ]:
def ver_tokens(texto):
    ids = tokenizador.encode(texto)
    pedazos = [tokenizador.decode([i]) for i in ids]
    print(f"{texto!r:35} → {len(ids)} tokens: {pedazos}")

ver_tokens("Monday")
ver_tokens("machine learning")
ver_tokens("hola")
ver_tokens("murciélago")

> 📝 El símbolo `' learning'` lleva un **espacio delante**: en GPT-2 el espacio va pegado al inicio de la palabra. Para el modelo, `"learning"` y `" learning"` son tokens **distintos**.

### 🔮 Ejercicio 2 — Predice cuántos tokens
¿Cuántos tokens crees que usará GPT-2 para cada una de estas palabras? Escribe tus apuestas y luego ejecuta.
- `"Electroencephalography"` (inglés)
- `"electroencefalografista"` (español)

✍️ *Mis apuestas:*

In [ ]:
ver_tokens("Electroencephalography")
ver_tokens("electroencefalografista")
ver_tokens("The cat sleeps on the sofa")
ver_tokens("El gato duerme en el sofá")

<details><summary><b>👀 Ver respuesta</b></summary>

- `Electroencephalography` → **5** tokens · `electroencefalografista` → **7** tokens.
- La frase en inglés usa **6** tokens (uno por palabra); la misma frase en español usa **11**, casi el doble, porque *gato*, *duerme* y *sofá* se parten en pedazos.

Consecuencia práctica: el mismo texto en español "cuesta" más tokens que en inglés para un tokenizador entrenado con inglés. Como los LLM cobran y limitan por tokens (Lección 13.9), eso importa. Los tokenizadores más recientes están entrenados con muchos idiomas y el problema es menor.
</details>

### 📖 Tokens → números
Cada token tiene un **número de identificación** (ID) en el vocabulario. El modelo nunca ve letras: ve una **secuencia de IDs**.

In [ ]:
ids = tokenizador.encode("The cat sleeps")
print("IDs:", ids)
print("Tamaño del vocabulario de GPT-2:", tokenizador.vocab_size)

### ✋ Ejercicio 3
El token `" cat"` tiene el ID 3,797 y `" sleeps"` el 44,263. ¿Significa eso que "sleeps" es "12 veces más" que "cat", o que están "lejos" en significado?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** El ID es solo un número de lista, como el número de una casilla en un casillero. Es **arbitrario**: no dice nada del significado. Si usáramos los IDs directamente como números, el modelo creería que el token 3,798 se parece al 3,797 solo por estar al lado. Necesitamos otra representación… 👇
</details>

---
## 3. 📖 El problema del *one-hot*

En la Lección 2.3 representabas categorías con **one-hot**: un vector lleno de ceros con un solo 1. Con palabras sería así:

```
perro  = [1, 0, 0, 0, 0, ...]
gato   = [0, 1, 0, 0, 0, ...]
camión = [0, 0, 1, 0, 0, ...]
```

Para medir qué tan parecidas son dos palabras usaremos la **similitud coseno**: compara hacia dónde "apuntan" dos vectores. Vale **1** si apuntan igual, **0** si no tienen nada en común y **−1** si son opuestos.

### 🔮 Ejercicio 4 — ¿Quién se parece a quién?
Con one-hot, ¿qué similitud tendrán *perro*–*gato*? ¿Y *perro*–*camión*?

✍️ *Mi predicción:*

In [ ]:
palabras = ["perro", "gato", "camión"]
one_hot = torch.eye(len(palabras))           # una fila por palabra, con un único 1

def similitud(a, b):
    return F.cosine_similarity(a, b, dim=0).item()

print(f"perro–gato  : {similitud(one_hot[0], one_hot[1]):.2f}")
print(f"perro–camión: {similitud(one_hot[0], one_hot[2]):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos dan **0**. Con one-hot **todas las palabras están igual de lejos de todas**: el modelo no tiene ninguna pista de que perro y gato se parecen. Además, con un vocabulario de 50,000 palabras cada vector tendría 50,000 números… casi todos ceros.

Necesitamos vectores **cortos** y **densos** (con números distintos de cero), donde la posición en el espacio **signifique algo**.
</details>

---
## 4. 📖 El *embedding*: una tabla de direcciones

Un *embedding* es simplemente una **tabla**: una fila por token, con unos pocos números por fila. Para conseguir el vector de un token, **buscas su fila** usando su ID.

Hagámoslo a mano con 3 dimensiones que **inventamos** nosotros:

In [ ]:
#                       ¿es animal?  ¿se come?  ¿tiene motor?
tabla = np.array([
    [0.9,         0.1,       0.0],     # 0: perro
    [0.8,         0.1,       0.0],     # 1: gato
    [0.0,         0.9,       0.0],     # 2: manzana
    [0.1,         0.0,       0.9],     # 3: camión
])
vocabulario = {"perro": 0, "gato": 1, "manzana": 2, "camión": 3}

def coseno(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

v = lambda palabra: tabla[vocabulario[palabra]]      # "buscar la fila"
print(f"perro–gato    : {coseno(v('perro'), v('gato')):.2f}")
print(f"perro–manzana : {coseno(v('perro'), v('manzana')):.2f}")
print(f"perro–camión  : {coseno(v('perro'), v('camión')):.2f}")

### ✏️ Ejercicio 5 — Agrega una palabra
Añade **"caballo"** a la tabla (fila 4) y al vocabulario, con los números que te parezcan razonables. Luego calcula su similitud con *perro* y con *camión*. ¿Qué números le pusiste?

In [ ]:
# ✏️ Completa: agrega una fila a la tabla y el nuevo ID al vocabulario
tabla_2 = np.vstack([tabla, [0.9, 0.0, 0.1]])      # ✏️ caballo: ¿es animal? ¿se come? ¿tiene motor?
vocabulario["caballo"] = 4

print(f"caballo–perro : {coseno(tabla_2[4], tabla_2[0]):.2f}")
print(f"caballo–camión: {coseno(tabla_2[4], tabla_2[3]):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `[0.9, 0.0, 0.1]` (animal, no se come, casi sin motor), sale **≈0.99** con perro y **≈0.21** con camión. Cualquier elección parecida da el mismo patrón.

El problema de este enfoque: **inventar las dimensiones a mano** no escala. ¿Quién escribe "¿es animal?" para 50,000 palabras? La gran idea es que **la red aprenda la tabla sola**, igual que aprende cualquier otro peso.
</details>

### 📖 En PyTorch: `nn.Embedding`
`nn.Embedding(num_tokens, dimensiones)` es esa tabla, pero con números **aleatorios al principio** que se ajustan con retropropagación (Lección 9.4). Se usa como cualquier capa: le das IDs y te devuelve vectores.

In [ ]:
capa = nn.Embedding(num_embeddings=4, embedding_dim=3)    # 4 tokens, 3 números cada uno
ids = torch.tensor([0, 3, 0])                             # "perro camión perro"
print(capa(ids))
print("Forma del resultado:", tuple(capa(ids).shape))

### 🔮 Ejercicio 6 — Cuenta los parámetros
1. En la salida de arriba, ¿por qué la primera y la tercera fila son **idénticas**?
2. GPT-2 tiene un vocabulario de 50,257 tokens y vectores de 768 números. ¿Cuántos parámetros tiene **solo** su tabla de *embeddings*?

✍️ *Mis respuestas:*

In [ ]:
tabla_gpt2 = gpt2.transformer.wte.weight          # wte = "word token embeddings"
print("Forma de la tabla de GPT-2:", tuple(tabla_gpt2.shape))
print(f"Parámetros: {tabla_gpt2.numel():,}  (de {sum(p.numel() for p in gpt2.parameters()):,} en todo el modelo)")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Porque las dos son el token 0 (*perro*): la capa **busca la misma fila**. Un *embedding* es una búsqueda en una tabla, nada más.
2. 50,257 × 768 = **38,597,376** parámetros: ¡casi un tercio de los ~124 millones de GPT-2 small! El vocabulario es caro.
</details>

---
## 5. 📖 Cómo se aprende el mapa: "dime con quién andas…"

> *"Conocerás una palabra por las compañías que frecuenta."* — J. R. Firth (lingüista, 1957)

Es la misma idea que KNN (Lección 5.3): **dime con quién andas y te diré quién eres**. Si *perro* y *gato* aparecen rodeados de las mismas palabras ("mi ___ duerme en casa", "el ___ corre en el parque"), deben significar algo parecido.

El método **word2vec** (*skip-gram*) convierte esa idea en un problema de clasificación:

> Dada una palabra, **predice qué palabras aparecen cerca** (dentro de una **ventana** de ±2 palabras).

Para acertar, la red se ve obligada a darle vectores parecidos a palabras que tienen vecinos parecidos. **Los *embeddings* son un subproducto** de esa tarea.

Primero escribimos un pequeño corpus: frases sobre animales, frutas y vehículos.

In [ ]:
rng = np.random.default_rng(0)
grupos = {
    "animal":   ["perro", "gato", "caballo", "conejo"],
    "fruta":    ["manzana", "pera", "uva", "naranja"],
    "vehiculo": ["coche", "camion", "autobus", "taxi"],
}
plantillas = {
    "animal":   ["el {} duerme en casa", "mi {} corre en el parque", "el {} come y duerme",
                 "le doy agua a mi {}", "el {} juega en el parque"],
    "fruta":    ["la {} es dulce", "como una {} madura", "la {} esta madura y dulce",
                 "compro {} en el mercado", "una {} fresca del mercado"],
    "vehiculo": ["el {} tiene ruedas", "manejo el {} por la carretera", "el {} necesita gasolina",
                 "estaciono el {} en la calle", "el {} va rapido por la carretera"],
}
frases = []
for _ in range(400):
    g = rng.choice(list(grupos))
    frases.append(rng.choice(plantillas[g]).format(rng.choice(grupos[g])))

vocab = sorted({w for f in frases for w in f.split()})
w2i = {w: i for i, w in enumerate(vocab)}
print(f"{len(frases)} frases, {len(vocab)} palabras distintas. Algunas frases:")
for f in frases[:5]: print("  ·", f)

### ✋ Ejercicio 7 — Arma los pares a mano
El *skip-gram* convierte cada frase en pares **(palabra, vecina)**. Con una ventana de **±1**, escribe todos los pares de la frase `"el gato duerme"`.

✍️ *Mis pares:*

Luego ejecuta la celda (que arma los pares con ventana ±2 para todo el corpus) y compara con las primeras líneas.

In [ ]:
VENTANA = 2      # cuántas palabras a cada lado cuentan como "vecinas"

def armar_pares(frases, ventana):
    pares = []
    for f in frases:
        ws = f.split()
        for i, w in enumerate(ws):
            for j in range(max(0, i - ventana), min(len(ws), i + ventana + 1)):
                if j != i:
                    pares.append((w2i[w], w2i[ws[j]]))
    return torch.tensor(pares)

pares = armar_pares(frases, VENTANA)
print(f"{len(pares)} pares de entrenamiento. Los de la primera frase ({frases[0]!r}):")
print([(vocab[a], vocab[b]) for a, b in pares[:12].tolist()])

<details><summary><b>👀 Ver respuesta</b></summary>

Con ventana ±1, `"el gato duerme"` da **4 pares**: (el, gato), (gato, el), (gato, duerme), (duerme, gato). Cada palabra se empareja con su vecina de cada lado. Con ventana ±2 salen más pares por frase, porque cada palabra "ve" más lejos.
</details>

### 📖 La red: una tabla de *embeddings* y una capa de salida
- **Entrada:** el ID de una palabra → `nn.Embedding` → su vector (10 números).
- **Salida:** una capa lineal que, a partir de ese vector, da un puntaje para cada palabra del vocabulario: "¿qué tan probable es que sea vecina?".
- **Pérdida:** entropía cruzada (Lección 3.2), como en cualquier clasificador.

In [ ]:
def entrenar_embeddings(pares, dim=10, epocas=30, semilla=0):
    torch.manual_seed(semilla)
    tabla = nn.Embedding(len(vocab), dim)           # ← lo que nos interesa
    salida = nn.Linear(dim, len(vocab))             # ← solo sirve para entrenar
    opt = torch.optim.Adam(list(tabla.parameters()) + list(salida.parameters()), lr=0.01)
    for epoca in range(epocas):
        orden = torch.randperm(len(pares))
        for k in range(0, len(pares), 256):
            lote = pares[orden[k:k + 256]]
            perdida = F.cross_entropy(salida(tabla(lote[:, 0])), lote[:, 1])
            opt.zero_grad(); perdida.backward(); opt.step()
    print(f"Pérdida final: {perdida.item():.2f}")
    return tabla.weight.detach()

E = entrenar_embeddings(pares)

def vecinos(palabra, E, k=4):
    En = F.normalize(E, dim=1)                       # para que el producto sea la similitud coseno
    sims = En @ En[w2i[palabra]]
    mejores = sims.argsort(descending=True)[1:k + 1]  # [1:] para saltarnos a la palabra misma
    return [(vocab[i], round(sims[i].item(), 2)) for i in mejores]

### 🔮 Ejercicio 8 — ¿Quiénes son los vecinos?
Nadie le dijo a la red qué es un animal o una fruta. Aun así, ¿qué 4 palabras crees que estarán más cerca de **"perro"**? ¿Y de **"manzana"**?

✍️ *Mi predicción:*

In [ ]:
for palabra in ["perro", "manzana", "coche"]:
    print(f"{palabra:>8} → {vecinos(palabra, E)}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **perro** → gato, caballo, conejo (≈0.9) y luego alguna palabra de su contexto.
- **manzana** → pera, uva, naranja.
- **coche** → autobús, camión, taxi (≈0.8–0.98).

¡La red **descubrió las categorías sola**, solo mirando con qué palabras aparece cada una! Es el mismo truco, a gran escala, que usan todos los modelos de lenguaje.
</details>

### ▶️ Veamos el mapa
Los vectores tienen 10 dimensiones; con PCA (Lección 8.4) los aplastamos a 2 para poder dibujarlos.

In [ ]:
def dibujar_mapa(E, palabras, titulo):
    coords = PCA(n_components=2, random_state=0).fit_transform(E[[w2i[p] for p in palabras]].numpy())
    colores = {p: c for g, c in zip(grupos, ["tab:orange", "tab:green", "tab:blue"]) for p in grupos[g]}
    plt.scatter(coords[:, 0], coords[:, 1], c=[colores.get(p, "gray") for p in palabras], s=60)
    for (x, y), p in zip(coords, palabras):
        plt.annotate(p, (x, y), textcoords="offset points", xytext=(4, 4))
    plt.title(titulo); plt.xticks([]); plt.yticks([]); plt.show()

palabras_mapa = sum(grupos.values(), []) + ["parque", "mercado", "carretera", "dulce", "gasolina"]
dibujar_mapa(E, palabras_mapa, "Mapa de significados aprendido (PCA a 2D)")

### ✋ Ejercicio 9
1. ¿Forman barrios los tres grupos de colores?
2. Las palabras grises ("parque", "mercado", "gasolina"…) aparecen **junto a** los animales, las frutas y los vehículos. ¿Quedaron **dentro** de esos barrios? ¿Por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Sí: animales, frutas y vehículos quedan en tres zonas separadas.
2. **No.** Quedan **fuera** de los barrios ("mercado" y "dulce" se van lejos, a un lado; "parque" y "gasolina" quedan entre grupos). Es una sutileza importante: word2vec junta palabras que **tienen las mismas vecinas** (que se pueden **intercambiar** en una frase: "mi ___ corre" admite perro, gato, conejo…), no palabras que **aparecen juntas**. "Perro" y "parque" van juntos en las frases, pero cumplen papeles distintos, así que no se parecen. (Las posiciones exactas pueden variar un poco en tu máquina; lo importante es el patrón.)
</details>

---
## 6. 📖 Los *embeddings* de un modelo real: GPT-2

GPT-2 aprendió su tabla de 50,257 × 768 leyendo ~40 GB de texto de internet. Vamos a explorar su mapa con las mismas herramientas.

> 📝 Usamos palabras en **inglés** con un **espacio delante** (`" dog"`), porque así es como GPT-2 las vio casi siempre y cada una es **un solo token**.

In [ ]:
E_gpt2 = gpt2.transformer.wte.weight.detach()
E_gpt2_norm = F.normalize(E_gpt2, dim=1)

def vector(palabra):
    ids = tokenizador.encode(palabra)
    assert len(ids) == 1, f"{palabra!r} no es un solo token: {ids}"
    return E_gpt2[ids[0]]

def vecinos_gpt2(v, k=6, excluir=()):
    sims = E_gpt2_norm @ F.normalize(v, dim=0)
    resultado = []
    for i in sims.topk(k + 10).indices.tolist():
        tok = tokenizador.decode([i])
        if tok not in excluir:
            resultado.append((tok, round(sims[i].item(), 2)))
    return resultado[:k]

for palabra in [" Monday", " dog", " red"]:
    print(f"{palabra!r:>9} → {vecinos_gpt2(vector(palabra), excluir=(palabra,))}")

### 🔮 Ejercicio 10 — Pon a prueba el mapa
Antes de ejecutar, apuesta: ¿cuáles serán los vecinos más cercanos de `" five"`? ¿Y de `" Paris"`?

✍️ *Mis apuestas:*

In [ ]:
for palabra in [" five", " Paris"]:          # ✏️ prueba con otras: " happy", " king", " January"…
    print(f"{palabra!r:>9} → {vecinos_gpt2(vector(palabra), excluir=(palabra,))}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `" five"` → `" four"`, `" six"`, `" three"`, `" seven"`… (similitud ≈0.85–0.88): los números viven juntos.
- `" Paris"` → `"Paris"` (sin espacio), `" France"`, `" French"`, `" London"`… Mezcla dos tipos de cercanía: **"cosas relacionadas con París"** (Francia, francés) y **"cosas del mismo tipo"** (otras capitales).
</details>

### 📖 Aritmética de significados
Si el mapa tiene sentido, las **direcciones** también deberían significar algo. La "flecha" que va de *man* (hombre) a *woman* (mujer) debería parecerse a la que va de *king* (rey) a *queen* (reina):

```
king − man + woman ≈ ?
```

### 🔮 Ejercicio 11 — La analogía más famosa
¿Qué palabra crees que saldrá? (Excluimos las tres palabras de la pregunta.)

✍️ *Mi apuesta:*

In [ ]:
def analogia(a, b, c):
    # "a es a b como ? es a c"  →  a − b + c
    v = vector(a) - vector(b) + vector(c)
    return vecinos_gpt2(v, k=4, excluir=(a, b, c))

print("king − man + woman  →", analogia(" king", " man", " woman"))
print("walking − walk + swim →", analogia(" walking", " walk", " swim"))   # ✏️ inventa la tuya

<details><summary><b>👀 Ver respuesta</b></summary>

- `king − man + woman` → **`" queen"`** (≈0.71), seguida de `" princess"`.
- `walking − walk + swim` → **`" swimming"`** (≈0.73).

Las direcciones del mapa codifican relaciones: "versión femenina de", "forma en -ing de"… **Ojo:** no siempre funciona tan bien; los ejemplos famosos están algo escogidos. Prueba con otras analogías (`" Paris"`, `" France"`, `" Italy"`) y verás que a veces el resultado es aproximado (sale `"Italy"` o `" Milan"` antes que `" Rome"`).
</details>

---
## 7. 📖 Qué pasa si… una palabra tiene varios significados

En estos *embeddings* cada token tiene **una sola fila**, sin importar la frase. Por eso se llaman ***embeddings* estáticos**. ¿Qué pasa con:

- *"Me senté en el **banco** del parque"* 🪑
- *"Pedí un préstamo en el **banco**"* 🏦

Las dos frases usan **el mismo vector**: un promedio raro de "asiento" y "dinero". El modelo no puede distinguirlas solo con la tabla.

La solución llegó con los modelos que leen la frase completa: **redes recurrentes** (Lección 11.3) y sobre todo **Transformers** (Módulo 12), que **ajustan** el vector de cada palabra según su contexto (*embeddings* **contextuales**). La tabla de esta lección es solo el **punto de partida** de esos modelos.

### ✋ Ejercicio 12
Busca los vecinos de `" bank"` en GPT-2. ¿Aparecen palabras del significado "orilla de un río" (*river*, *shore*)? ¿Por qué crees que gana un significado?

In [ ]:
print(vecinos_gpt2(vector(" bank"), k=8, excluir=(" bank",)))

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Salen palabras de **finanzas** (`" banks"`, `" Bank"`, `" banking"`…), no de ríos. En el texto de internet "bank" aparece muchísimo más como entidad financiera, así que ese significado **domina** el vector. El significado minoritario queda casi invisible en la tabla estática.

⚠️ Lo mismo pasa con los **sesgos**: si en los textos ciertas profesiones aparecen casi siempre junto a un género, el mapa lo aprende y lo reproduce (Lección 15.4). Los *embeddings* son un espejo de los datos, con todo lo bueno y lo malo.
</details>

---
## 8. 📖 Hiperparámetros y perillas

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `embedding_dim` (dimensiones) | Cuántos números describen cada token | Más matices, pero más parámetros y más datos necesarios | Mapa "aplastado": se mezclan significados | 50–300 (word2vec); 768+ en LLMs |
| Ventana (*window*) | Cuántas vecinas cuentan como contexto | Capta **temas** (perro ~ parque) | Capta **función gramatical** y sinónimos | 2–5 |
| Tamaño del vocabulario | Cuántos tokens distintos existen | Secuencias más cortas, tabla más grande | Más pedazos por palabra | 30k–100k subpalabras |
| Épocas / datos | Cuánto se entrena | Mapa más estable (hasta saturar) | Mapa ruidoso | Más datos > más épocas |

### ✏️ Ejercicio 13 — Un mapa demasiado pequeño
Vuelve a entrenar con `dim=1` (¡un solo número por palabra!) y mira los vecinos. 🔮 Antes de ejecutar: ¿seguirán separados animales, frutas y vehículos?

In [ ]:
DIM = 1     # ✏️ prueba 1, 2, 10, 50
E_chico = entrenar_embeddings(pares, dim=DIM)
for palabra in ["perro", "manzana", "coche"]:
    print(f"{palabra:>8} → {vecinos(palabra, E_chico)}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `dim=1` todo se rompe: cada palabra es un solo número, así que la similitud coseno solo puede valer **+1 o −1** (mismo signo o signo contrario). Los "vecinos" salen todos con 1.0 y mezclados entre grupos, y la pérdida final queda más alta que con 10 dimensiones. Con 2 dimensiones ya mejora, y con 10 los tres grupos quedan limpios. Con 50 no ganas nada visible: para 3 categorías y 50 palabras sobra espacio.

Moraleja: la dimensión es un compromiso. Muy pocas → el mapa no cabe; demasiadas → desperdicias parámetros y necesitas más datos.
</details>

---
## 9. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Muchas palabras se convierten en `<desconocida>` | Tokenizador por palabras con vocabulario cerrado | Tokenización por subpalabras |
| El modelo funciona peor en español que en inglés | Tokenizador entrenado con inglés: más tokens por palabra | Tokenizador / modelo multilingüe |
| Los vecinos de una palabra no tienen sentido | Pocos datos, pocas épocas o dimensión demasiado pequeña | Más texto; subir `embedding_dim` |
| `"Paris"` y `" Paris"` dan resultados distintos | Son tokens distintos (el espacio cuenta) | Tokeniza igual que el modelo lo vio al entrenar |
| Una palabra con dos significados "mezcla" ambos | *Embeddings* estáticos | Modelos con contexto (11.3–11.5, M12) |
| El mapa reproduce estereotipos | Sesgos del texto de entrenamiento | Auditar (Lección 15.4) |

### 🔮 Ejercicio 14
Entrenaste *embeddings* con noticias de deportes. Ahora quieres usarlos en un clasificador de **recetas de cocina**. ¿Qué problemas esperas?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Vocabulario:** muchas palabras de cocina (*sofreír*, *cúrcuma*) no aparecieron nunca en noticias deportivas: no tienen vector (o lo tienen malo).
2. **Significados:** palabras compartidas significan otra cosa en cada dominio ("batir un récord" vs. "batir huevos", "pase" vs. "pase por harina"). El mapa reflejará el significado deportivo.

Es un problema de **cambio de dominio**, parecido al *transfer learning* de la Lección 10.5: un modelo preentrenado ayuda, pero conviene ajustarlo (*fine-tuning*) con datos del nuevo dominio.
</details>

---
## 10. 📋 Ficha: *embeddings* de palabras

| | *Embeddings* (word2vec / capa `nn.Embedding`) |
|---|---|
| **Qué es** | Una tabla: cada token → un vector denso de pocas dimensiones |
| **Intuición** | La dirección de cada palabra en un mapa de significados |
| **Cómo aprende** | Como pesos de una red, prediciendo el contexto ("dime con quién andas") o como primera capa de un modelo más grande |
| **Hiperparámetros clave** | `embedding_dim`, ventana, tamaño del vocabulario |
| **✅ Fortalezas** | Captura similitudes y relaciones · Mucho más compacto que one-hot · Reutilizable (preentrenado) |
| **❌ Debilidades** | Un vector por token (estático) · Hereda sesgos · Depende del tokenizador y del dominio |
| **Úsalo cuando…** | Cualquier modelo que procese texto (o categorías con muchos valores: productos, usuarios…) |

## 11. 🧠 Resumen en 5 puntos
1. Un modelo de lenguaje no ve letras: ve **tokens** (normalmente **subpalabras**) convertidos a **IDs**.
2. Los IDs y el **one-hot** no dicen nada del significado: con one-hot todas las palabras están igual de lejos.
3. Un ***embedding*** es una **tabla** que da a cada token un vector denso: su dirección en un **mapa de significados**.
4. La tabla se **aprende**: prediciendo el contexto, las palabras que "andan con las mismas" terminan cerca. Hasta las direcciones significan algo (*king − man + woman ≈ queen*).
5. Son **estáticos** (un vector por token, sin importar la frase) y heredan los **sesgos** del texto; los modelos de las próximas lecciones los usan como punto de partida y les añaden contexto.

---
## 12. 🏋️ Ejercicios finales

### 12.1 Repaso rápido (opción múltiple)

**1.** ¿Por qué los LLM usan tokens de **subpalabras**?
- a) Porque son más bonitos · b) Para tener un vocabulario manejable y nunca encontrar una palabra "desconocida" · c) Porque las letras no se pueden representar · d) Para que el texto ocupe más

**2.** Con *one-hot*, la similitud coseno entre dos palabras distintas es siempre…
- a) 1 · b) 0.5 · c) 0 · d) −1

**3.** Una capa `nn.Embedding(10000, 64)`, ¿cuántos parámetros tiene?
- a) 10,064 · b) 640,000 · c) 64 · d) 10,000

**4.** En word2vec, ¿por qué *perro* y *gato* terminan con vectores parecidos?
- a) Porque alguien los etiquetó como animales · b) Porque empiezan con letras parecidas · c) Porque aparecen rodeados de palabras parecidas · d) Por casualidad

**5.** ¿Cuál es la principal limitación de un *embedding* estático?
- a) Ocupa demasiado · b) Da el mismo vector a "banco" (asiento) y "banco" (dinero) · c) No funciona con PyTorch · d) Solo sirve en inglés

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Las frecuentes son un token; las raras se arman con pedazos.
2. **c)** Los vectores one-hot de palabras distintas no comparten ninguna posición con 1.
3. **b)** 10,000 filas × 64 columnas = 640,000.
4. **c)** "Dime con quién andas…": comparten contextos.
5. **b)** Un vector por token, sin importar el contexto.
</details>

### 12.2 Preguntas para pensar (sin ejecutar)

**A.** Una tienda en línea tiene 2 millones de productos y quiere recomendar "productos parecidos". ¿Cómo podrías aplicar la idea de word2vec **sin usar texto**? (Pista: ¿qué sería una "frase"? ¿Y una "palabra"?)

**B.** Un compañero dice: *"Para mi clasificador de sentimientos en español voy a usar los embeddings de GPT-2 tal cual, porque GPT-2 es enorme"*. ¿Qué dos problemas le señalarías, según lo que viste en esta lección?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Trata cada **sesión de compra** (o carrito) como una "frase" y cada **producto** como una "palabra". Los productos que se compran o se ven en las mismas sesiones terminarán cerca en el mapa. Esta idea (*item2vec* / *prod2vec*) se usa de verdad en recomendadores (Lección 8.7).

**B.** (1) El tokenizador de GPT-2 se entrenó sobre todo con inglés: corta las palabras en español en muchos pedazos sin significado claro, así que muchos vectores servirán poco. (2) Son *embeddings* **estáticos**: "no está mal" y "está mal" comparten casi los mismos vectores; el sentimiento depende del **orden y el contexto**, que la tabla sola no captura. Mejor: un modelo multilingüe con contexto (M12) o, al menos, entrenar una red de secuencias encima.
</details>

### 12.3 🧩 Práctica integradora: el detector de intrusos 🕵️

Vas a construir, con los *embeddings* de GPT-2, un juego clásico: dada una lista de palabras, **¿cuál no pertenece al grupo?**

**Paso 1 ✋ — Piensa el algoritmo.** Tienes la función `vector(palabra)` y la similitud coseno. Describe en una frase cómo detectarías al intruso.

✍️ *Mi idea:*

<details><summary><b>👀 Ver respuesta</b></summary>

Para cada palabra, calcula su **similitud promedio con las demás**. El intruso es la palabra con el promedio **más bajo**: la que vive más lejos del barrio de las otras.
</details>

**Paso 2 ▶️ — La matriz de similitudes.** Ejecuta para ver la similitud de cada par.

In [ ]:
grupo = [" Monday", " Tuesday", " banana", " Friday", " Sunday"]

V = F.normalize(torch.stack([vector(p) for p in grupo]), dim=1)
matriz = (V @ V.T).numpy()

plt.imshow(matriz, cmap="viridis"); plt.colorbar()
plt.xticks(range(len(grupo)), grupo, rotation=45); plt.yticks(range(len(grupo)), grupo)
plt.title("Similitud coseno entre las palabras"); plt.show()

🔮 ¿Qué fila de la matriz se ve "más apagada"?

<details><summary><b>👀 Ver respuesta</b></summary>

La de `" banana"`: su similitud con los días ronda 0.3, mientras que entre días es ≈0.8–0.9.
</details>

**Paso 3 ▶️ — El detector.** Ahora el algoritmo completo.

In [ ]:
def intruso(palabras):
    V = F.normalize(torch.stack([vector(p) for p in palabras]), dim=1)
    sim = V @ V.T
    promedio = (sim.sum(1) - 1) / (len(palabras) - 1)    # quitamos la similitud consigo misma (= 1)
    for p, s in zip(palabras, promedio):
        print(f"   {p!r:>12}: {s:.2f}")
    return palabras[promedio.argmin()]

print("Intruso:", intruso(grupo))

**Paso 4 ✏️ — Ponlo a prueba.** Prueba con estos grupos (y con los tuyos). ¿Acierta siempre?

In [ ]:
pruebas = [
    [" dog", " cat", " horse", " car"],
    [" red", " blue", " green", " happy"],
    [" apple", " orange", " banana", " grape"],     # ✏️ ¿hay intruso aquí?
]
for p in pruebas:
    print("Intruso:", intruso(p), "\n")

✋ ¿Qué pasó con el tercer grupo? ¿Es un error del detector?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

En los dos primeros acierta (`" car"` y `" happy"`). En el tercero **no hay intruso**, pero el detector siempre elige uno: el de menor promedio. Suele salir `" orange"`, porque también es un **color** y su vector está "repartido" entre los dos significados (¡el problema de los *embeddings* estáticos!). Un detector serio necesitaría un **umbral**: "solo declaro intruso si su promedio es mucho menor que el de los demás".
</details>

**Paso 5 ✋ — En español.** La función `vector` exige un solo token. Prueba `vector(" perro")`. ¿Qué pasa y cómo lo arreglarías?

In [ ]:
try:
    vector(" perro")
except AssertionError as error:
    print("⚠️", error)

✍️ *Mi arreglo:*

<details><summary><b>👀 Ver respuesta</b></summary>

`" perro"` se corta en 2 tokens (`' per'` + `'ro'`), así que no hay una sola fila de la tabla para él. Arreglo simple: **promediar** los vectores de sus pedazos. Funciona regular, porque `' per'` aparece en muchas palabras inglesas (*person*, *perfect*) y arrastra su significado. Arreglo de verdad: usar un modelo con un tokenizador multilingüe, o uno con contexto, que combine los pedazos de forma inteligente (Módulo 12).
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Embedding Projector* de TensorFlow (projector.tensorflow.org): explora word2vec en 3D.
- 📘 **Artículo:** Jay Alammar, *The Illustrated Word2vec* (inglés, muy visual).
- 🎮 **Interactivo:** *Tiktokenizer* (tiktokenizer.vercel.app): mira cómo cortan el texto distintos tokenizadores.
- 📺 **Video:** StatQuest, "Word Embedding and Word2Vec, Clearly Explained" (inglés, con subtítulos).

---
⬅️ **Anterior:** [11.1 Datos con orden: texto, audio y series de tiempo](11.1-datos-con-orden.ipynb) · ➡️ **Siguiente:** [11.3 Redes recurrentes (RNN) y su memoria corta](11.3-redes-recurrentes.ipynb)